In [1]:
import pandas as pd
from pathlib import Path

#read input files 
IO_file = Path(r"C:\Users\menne\Documents\Master\Thesis\Data\MRIO\Figaro\Industry\flatfile_eu-ic-io_ind-by-ind_25ed_2023\flatfile_eu-ic-io_ind-by-ind_25ed_2023.csv") #input output tables for industry
GHG_file = Path(r"C:\Users\menne\Documents\Master\Thesis\Data\MRIO\Figaro\Emissions\env_ac_ghgfp_2023_cleaned.csv") #Emissions 

df_IO = pd.read_csv(IO_file, sep=",")
df_GHG = pd.read_csv(GHG_file, sep=";", encoding="utf-8-sig", low_memory=False)


In [3]:
# ADD 4 COUNTRIES TO REST OF WORLD

countries_to_row = ["AL", "ME", "MK", "RS"]

for column in ["refArea", "counterpartArea"]:
    df_IO[column] = df_IO[column].replace(
        {country: "FIGW1" for country in countries_to_row}
    )

df_IO["obsValue"] = pd.to_numeric(
    df_IO["obsValue"],
    errors="coerce"
).fillna(0)

df_IO = (
    df_IO
    .groupby(
        ["refArea", "rowIi", "counterpartArea", "colIi"],
        as_index=False,
        dropna=False
    )["obsValue"]
    .sum()
)



df_IO["icioiRow"] = (
    df_IO["refArea"].astype(str)
    + "_"
    + df_IO["rowIi"].astype(str)
)

df_IO["icioiCol"] = (
    df_IO["counterpartArea"].astype(str)
    + "_"
    + df_IO["colIi"].astype(str)
)

df_IO = df_IO[
    [
        "icioiRow",
        "icioiCol",
        "refArea",
        "rowIi",
        "counterpartArea",
        "colIi",
        "obsValue"
    ]
]